# Deploy and Test Inference Container

This notebook deploys the containerized SLM inference server and validates it meets performance requirements.

## What This Notebook Does

1. **Pulls Container Image**: Downloads from ACR to deployment target (local or edge device)
2. **Runs Container**: Starts inference server with resource constraints
3. **Health Checks**: Validates server is ready to accept requests
4. **Performance Testing**: Measures inference latency and throughput
5. **Deployment Strategies**: Demonstrates blue-green deployment and rollback procedures

## Why Container Deployment?

**Consistency:**
- Same image runs identically on dev laptop, cloud VM, or embedded device
- No "works on my machine" issues
- Dependencies packaged with application

**Portability:**
- Runs on any Linux system with Docker
- Easy migration between cloud providers or on-prem
- Simplified CI/CD pipelines

**Resource Management:**
- CPU and memory limits prevent resource exhaustion
- Container orchestration enables auto-scaling
- Efficient multi-tenancy on shared hardware

## Deployment Targets

### Local Testing (This Notebook)
- Docker Desktop or Docker Engine
- Full debugging capabilities
- No network latency considerations

### Edge/Embedded Devices
- Raspberry Pi 4/5 (ARM64, 4-8GB RAM)
- NVIDIA Jetson Nano/Xavier (ARM64, GPU optional)
- Intel NUC (x86_64, compact form factor)
- Industrial PCs with Linux

### Cloud Deployment
- Azure Container Instances (serverless)
- Azure Kubernetes Service (orchestrated)
- Azure App Service (managed containers)
- Self-managed VMs with Docker

## Performance Validation

### Latency Targets
- **p50 (median)**: <30ms for typical requests
- **p95**: <50ms (95% of requests complete within this time)
- **p99**: <100ms (99th percentile)

### Throughput Targets
- **Tokens/second**: 50-100 tokens/sec on CPU
- **Concurrent requests**: 5-10 with acceptable latency degradation
- **Max queue depth**: 20 requests before backpressure

### Resource Constraints
- **CPU**: 2-4 cores for optimal throughput
- **Memory**: <4GB peak usage (model + inference)
- **Disk**: Minimal I/O (all in-memory inference)

## Health Check Strategy

### Startup Checks
1. Container starts successfully
2. Model loads without errors
3. `/health` endpoint returns 200 OK
4. First inference request completes

### Runtime Monitoring
- Periodic health checks (every 30-60 seconds)
- Latency percentile tracking
- Memory usage monitoring
- Error rate tracking

### Failure Detection
- 3 consecutive health check failures → restart container
- Latency exceeds threshold → alert and investigate
- Memory limit reached → OOM kill and restart

## Deployment Strategies

### Blue-Green Deployment
1. Deploy new version (green) alongside current (blue)
2. Validate green version with synthetic traffic
3. Switch router/load balancer to green
4. Keep blue running briefly for quick rollback
5. Decommission blue after validation period

### Rolling Update
1. Deploy new version to 1 instance
2. Monitor metrics for degradation
3. If healthy, roll out to remaining instances
4. If issues detected, halt and rollback

### Canary Deployment
1. Route 5% of traffic to new version
2. Compare metrics to baseline
3. Gradually increase traffic percentage
4. Rollback if anomalies detected

## Rollback Procedures

**When to Rollback:**
- Increased error rate (>1% of requests)
- Latency degradation (>20% increase in p95)
- Memory leaks (continuous growth)
- Failed health checks

**How to Rollback:**
1. Stop new container: `docker stop <container_id>`
2. Start previous version: `docker start <previous_container_id>`
3. Verify health checks pass
4. Monitor metrics return to baseline
5. Investigate root cause offline

## Prerequisites

- Completed notebook `11-push-to-acr.ipynb`
- Container image available in ACR
- Docker installed on deployment target
- Network access to ACR
- Sufficient resources (4GB RAM, 2+ CPU cores recommended)

## Expected Duration

- Image pull: ~5-10 minutes (first time, cached after)
- Container startup: ~30-60 seconds (model loading)
- Validation: ~5 minutes
- Total: ~15-20 minutes

## Production Considerations

**Security:**
- Run container as non-root user
- Use read-only filesystem where possible
- Mount secrets as environment variables or volumes
- Network segmentation and firewall rules

**Monitoring:**
- Export metrics to Prometheus/Grafana
- Centralized logging (ELK, Azure Monitor)
- Distributed tracing (OpenTelemetry)
- Alerting on SLA violations

**High Availability:**
- Run multiple replicas behind load balancer
- Health-based routing (remove unhealthy instances)
- Graceful shutdown with connection draining
- Persistent storage for checkpoints (if needed)

# Notebook 12: Deploy & Run Inference Container

Best-practice walkthrough for pulling, running, validating, and updating the SLM inference container.

## 0. Prerequisites
- Image pushed to ACR: `<registry>.azurecr.io/slm-inference:<version>`
- Local Docker available
- Port availability (default: `8080`)
- Adequate host memory for chosen model variant

## 1. Parameters
Define runtime configuration for container startup.

## 2. Pull Image & Inspect
Pull the tagged image and inspect size & layers.

## 3. Run Container with Resource Constraints
Demonstrate CPU and memory limits to simulate edge conditions.

## 4. Health & Readiness Checks
Call `/health` endpoint; optionally add retry logic.

## 5. Generate Sample Text
Send request to `/generate` and measure latency.

## 6. Collect Local Metrics
Use `metrics_collector` to wrap generation calls and gather percentile stats.

## 7. Update / Roll Forward Strategy
Launch new version side-by-side (different tag), verify health, then switch traffic.

## 8. Rollback Strategy
If latency or errors spike, stop new container and revert to previous tag.

## 9. Next Steps (Future Enhancements)
- Add `/metrics` endpoint (Prometheus format)
- Integrate OpenTelemetry tracing
- Add structured JSON logging & correlation IDs

---
Proceed with the cells below.